In [0]:
# Standard library imports
import ast
import json
import warnings
from datetime import datetime, timedelta
from itertools import product
from pathlib import Path

# Visualization and data handling
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Time-series modeling
import statsmodels.api as sm
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Machine learning
from joblib import Memory
from scipy.stats import loguniform
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier


seed = 24601

google_analytics = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_google_analytics_abandonment.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

sales = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_sales.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

materials = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_material.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

customer = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_customer.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

cutoff_times = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_cutoff_times.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

operating_hours = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_operating_hours.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

orders = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_orders.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

visit_plan = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_visit_plan.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

# Standardize column names to lowercase across all dataframes
for df in [
    customer,
    cutoff_times,
    google_analytics,
    materials,
    operating_hours,
    orders,
    sales,
    visit_plan,
]:
    df.columns = df.columns.str.lower()

In [0]:
display(google_analytics.head(3))
display(sales.head(3))
display(materials.head(3))
display(customer.head(3))
display(cutoff_times.head(3))
display(operating_hours.head(3))
display(orders.head(3))
display(visit_plan.head(3))

# Helper Functions

In [0]:
# Define helper to detect channel switching
def switched(row):
    return not set(row["after_order_types"]).issubset(set(row["before_order_types"]))


# Preparation

[Describe the coming changes and how they will help the modeling stage]

In [0]:
ga_with_abandoned = google_analytics.copy()

# Fill missing abandoned flags with False
ga_with_abandoned["abandoned"] = (
    ga_with_abandoned["abandoned"]
    .astype(bool)
    .fillna(False)
)

# Sort by customer and event timestamp
ga_with_abandoned = ga_with_abandoned.sort_values(
    by=["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Identify customers who abandoned at least once
abandoned_customers = ga_with_abandoned.loc[
    ga_with_abandoned["abandoned"] == True, "customer_id"
].unique()

# Count unique abandoned customers
num_abandoned_customers = len(abandoned_customers)
print(f"Number of unique customers who abandoned: {num_abandoned_customers}")

# Filter to include only customers who have abandoned
ga_abandoned_customers = ga_with_abandoned[
    ga_with_abandoned["customer_id"].isin(abandoned_customers)
]

# Sort filtered data
ga_abandoned_customers = ga_abandoned_customers.sort_values(
    ["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Find the first abandonment timestamp per customer
first_abandoned_ts = (
    ga_abandoned_customers[ga_abandoned_customers["abandoned"] == True]
    .groupby("customer_id")["event_ts_utc"]
    .min()
    .reset_index()
    .rename(columns={"event_ts_utc": "first_abandoned_ts"})
)

# Merge first abandonment timestamp back to full table
ga_after_first_abandoned = ga_abandoned_customers.merge(
    first_abandoned_ts, on="customer_id", how="left"
)

# Keep only events after first abandonment
ga_after_first_abandoned = ga_after_first_abandoned[
    ga_after_first_abandoned["event_ts_utc"]
    >= ga_after_first_abandoned["first_abandoned_ts"]
].drop(columns=["first_abandoned_ts"])

# Sort by customer and timestamp
ga_after_first_abandoned = ga_after_first_abandoned.sort_values(
    ["customer_id", "event_ts_utc"], ascending=[True, True]
)

# Convert date columns to datetime
ga_after_first_abandoned["event_date"] = pd.to_datetime(
    ga_after_first_abandoned["event_date"]
)
orders["created_ts_utc"] = pd.to_datetime(orders["created_ts_utc"])
orders["created_date"] = orders["created_ts_utc"].dt.date
orders["created_date"] = pd.to_datetime(orders["created_date"])

# Build GA events table
ga_events = (
    ga_after_first_abandoned.loc[
        ga_after_first_abandoned["event_date"]
        >= ga_after_first_abandoned.groupby("customer_id")["event_date"].transform("min"),
        [
            "customer_id",
            "device_category",
            "device_mobile_brand_name",
            "device_operating_system",
            "event_date",
            "abandoned",
            "event_name",
        ],
    ]
    .assign(
        is_abandon=lambda df: df["abandoned"] == True,
        is_purchase=lambda df: df["event_name"].str.lower().str.contains("purchase"),
        is_order=False,
    )
)

# Build order events table
order_events = (
    orders[
        [
            "customer_id",
            "material_id",
            "sales_office_id",
            "order_quantity",
            "order_type",
            "created_date",
        ]
    ]
    .rename(columns={"created_date": "event_date"})
    .assign(
        device_category=None,
        device_mobile_brand_name=None,
        device_operating_system=None,
        abandoned=False,
        event_name="order_created",
        is_abandon=False,
        is_purchase=False,
        is_order=True,
    )
)

# Combine GA and order events
combined_events = pd.concat([ga_events, order_events], ignore_index=True)

# Sort combined events
combined_events = combined_events.sort_values(["customer_id", "event_date"]).reset_index(
    drop=True
)

# Assign order_type for purchases
combined_events.loc[combined_events["is_purchase"], "order_type"] = "mycoke360"

# Assign order_type for abandons
combined_events.loc[combined_events["is_abandon"], "order_type"] = "none"

# Merge combined events with customer table
combined_with_customer = combined_events.merge(
    customer,
    on="customer_id",
    how="left",
)

# Create dataframe of abandoned customers
abandoned_df = pd.DataFrame({"customer_id": abandoned_customers})

# Merge combined events with customer table again
combined_with_customer = combined_events.merge(
    customer,
    on="customer_id",
    how="left",
)

# Ensure all abandoned customers are retained
combined_with_customer = abandoned_df.merge(
    combined_with_customer,
    on="customer_id",
    how="left",
)

[Describe the coming changes and how they will help the modeling stage]

In [0]:
# Convert event_date to datetime
combined_with_customer["event_date"] = pd.to_datetime(combined_with_customer["event_date"])

# Compute first abandon date per customer
first_abandon_date = (
    combined_with_customer[combined_with_customer["is_abandon"] == True]
    .groupby("customer_id")["event_date"]
    .min()
    .reset_index()
    .rename(columns={"event_date": "first_abandon_date"})
)

# Merge first abandon date into main dataframe
combined_with_customer = combined_with_customer.merge(first_abandon_date, on="customer_id", how="left")

# Identify post-abandon purchases or orders
post_abandon = combined_with_customer[
    ((combined_with_customer["is_purchase"] == True) |
     (combined_with_customer["is_order"] == True)) &
    (combined_with_customer["event_date"] >= combined_with_customer["first_abandon_date"])
]

# Create churn flag based on absence of post-abandon activity
churn_flag = combined_with_customer[["customer_id"]].drop_duplicates().copy()
churn_flag["churned"] = ~churn_flag["customer_id"].isin(post_abandon["customer_id"])

# Calculate churn counts
churn_counts = churn_flag["churned"].value_counts()

# Calculate churn proportions
churn_proportions = churn_flag["churned"].value_counts(normalize=True)

# print("Counts:")
# print(churn_counts)
# print("\nProportions:")
# print(churn_proportions)

# Merge churn flag data into main dataframe
combined_with_customer = combined_with_customer.merge(
    churn_flag,
    on="customer_id",
    how="left"
)

[Describe the coming changes and how they will help the modeling stage]

In [0]:
# Keep events after first abandon
after_abandon = combined_with_customer[
    combined_with_customer['event_date'] >= combined_with_customer['first_abandon_date']
]

# Aggregate per customer
customer_flags = after_abandon.groupby('customer_id').agg(
    ordered_any=('is_order', 'any'),
    purchased_any=('is_purchase', 'any'),
    abandoned_any=('is_abandon', 'any')
).reset_index()

# Filter: ordered but never purchased or abandoned again
ordered_only_customers = customer_flags[
    (customer_flags['ordered_any'] == True) &
    (customer_flags['purchased_any'] == False)
]

print(f"Customers who ordered but never purchased or abandoned again: {len(ordered_only_customers)}")

[Describe the coming changes and how they will help the modeling stage]

In [0]:
# Split events into before and after first abandonment
before_abandon = combined_with_customer[
    combined_with_customer["event_date"] < combined_with_customer["first_abandon_date"]
]

after_abandon = combined_with_customer[
    combined_with_customer["event_date"] >= combined_with_customer["first_abandon_date"]
]

# Identify unique order types before abandonment
before_types = (
    before_abandon[before_abandon["is_order"]]
    .groupby("customer_id")["order_type"]
    .unique()
    .reset_index()
    .rename(columns={"order_type": "before_order_types"})
)

# Identify unique order types after abandonment
after_types = (
    after_abandon[after_abandon["is_order"]]
    .groupby("customer_id")["order_type"]
    .unique()
    .reset_index()
    .rename(columns={"order_type": "after_order_types"})
)

# Merge before and after order types
channel_switch = before_types.merge(after_types, on="customer_id", how="inner")

# Flag customers who switched order channels
channel_switch["switched_channel"] = channel_switch.apply(switched, axis=1)

# Filter switched customers
switched_customers = channel_switch[channel_switch["switched_channel"]]
print(f"Customers who switched order channel after first abandonment: {len(switched_customers)}")

[Describe the coming changes and how they will help the modeling stage]

In [0]:
# Build switch pairs for customers who changed order types
switch_pairs = []

for _, row in switched_customers.iterrows():
    before = set(row["before_order_types"])
    after = set(row["after_order_types"])
    new_channels = after - before

    # Record transitions from previous to new order types
    for b, a in product(before, new_channels):
        switch_pairs.append((b, a))

# Create dataframe of channel switches
switch_df = pd.DataFrame(switch_pairs, columns=["from_channel", "to_channel"])

# Summarize most frequent switch patterns
switch_summary = (
    switch_df.value_counts()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

# Modeling

In [0]:
# Aggregate behavioral features
customer_features = combined_with_customer.groupby('customer_id').agg(
    total_abandons=('is_abandon', 'sum'),
    total_purchases=('is_purchase', 'sum'),
    total_orders=('is_order', 'sum'),
    unique_devices=('device_category', 'nunique'),
    unique_mobile_brands=('device_mobile_brand_name', 'nunique'),
    unique_os=('device_operating_system', 'nunique'),
    avg_order_quantity=('order_quantity', 'mean')
).reset_index()

static_cols = [
    'customer_id', 'sales_office_location', 'cold_drink_channel_description',
    'customer_sub_trade_channel_description', 'distribution_mode',
    'shipping_duration', 'shipping_destination'
]

customer_features = customer_features.merge(
    combined_with_customer[static_cols].drop_duplicates('customer_id'),
    on='customer_id', how='left'
)

# adding target variable
customer_features = customer_features.merge(
    combined_with_customer[['customer_id', 'churned']].drop_duplicates('customer_id'),
    on='customer_id', how='left'
)
customer_features['churned'] = customer_features['churned'].astype(int)

# Encode categorical variables
categorical_cols = [
    "sales_office_location",
    "cold_drink_channel_description",
    "customer_sub_trade_channel_description",
    "distribution_mode",
    "shipping_duration",
    "shipping_destination",
]

for col in categorical_cols:
    le = LabelEncoder()
    customer_features[col] = le.fit_transform(customer_features[col].astype(str))

# Define features and target
X = customer_features.drop(["customer_id", "churned"], axis=1)
y = customer_features["churned"]

# Impute missing numeric values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# Initialize random forest model
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# Create stratified 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize metric trackers and importance accumulator
accuracy_scores, recall_scores, precision_scores, f1_scores, auc_scores = [], [], [], [], []
feature_importances = np.zeros(X.shape[1])

# Perform cross-validation loop
for train_idx, test_idx in skf.split(X_imputed, y):
    X_train, X_test = X_imputed[train_idx], X_imputed[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Collect performance metrics
    accuracy_scores.append(accuracy_score(y_test, y_pred))
    recall_scores.append(recall_score(y_test, y_pred))
    precision_scores.append(precision_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))
    auc_scores.append(roc_auc_score(y_test, y_prob))

    # Accumulate feature importances
    feature_importances += rf.feature_importances_

# Compute average feature importances
feature_importances /= skf.get_n_splits()

# Create feature importance dataframe
feat_imp_df = pd.DataFrame({
    "feature": X.columns,
    "importance": feature_importances
}).sort_values(by="importance", ascending=False)

# Print summary results
print("---- Cross-Validation Results ----")
print(f"Mean Accuracy:  {np.mean(accuracy_scores):.3f}")
print(f"Mean Recall:    {np.mean(recall_scores):.3f}")
print(f"Mean Precision: {np.mean(precision_scores):.3f}")
print(f"Mean F1 Score:  {np.mean(f1_scores):.3f}")
print(f"Mean ROC-AUC:   {np.mean(auc_scores):.3f}")

feat_imp_df["importance"] = feat_imp_df["importance"].map(lambda x: f"{x:.6f}")

[Interpretation]

In [0]:
# Prepare ranked feature-importance table
fi = feat_imp_df.copy()
fi = fi[["feature", "importance"]].dropna()
fi["feature"] = fi["feature"].astype(str)
fi["importance"] = pd.to_numeric(fi["importance"], errors="coerce")
fi = fi.dropna(subset=["importance"])

# Sort by importance and add a 1-based rank index
fi_sorted = fi.sort_values("importance", ascending=False).reset_index(drop=True)
fi_sorted.index = fi_sorted.index + 1
fi_table = fi_sorted.rename_axis("rank").reset_index()

# Display table (Databricks will render this like the image)
fi_table

# Plot feature importance to match the table order
plt.figure(figsize=(10, 6))
plt.barh(fi_sorted["feature"], fi_sorted["importance"], color="skyblue")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

[Interpretation]

In [0]:
# Summarize churn by sales office location
churn_summary = (
    combined_with_customer
    .groupby("sales_office_location")
    .agg(
        total_customers_who_abandoned=("customer_id", lambda x: x.nunique()),
        churned_customers=("customer_id", lambda x: x[combined_with_customer.loc[x.index, "churned"]].nunique()),
    )
    .reset_index()
)

# Calculate churn rate percentage
churn_summary["churn_rate"] = (
    churn_summary["churned_customers"] / churn_summary["total_customers_who_abandoned"]
) * 100

# Sort results by churn rate
churn_summary_sorted = churn_summary.sort_values("churn_rate", ascending=False)

# Plot churn rate by location
plt.figure(figsize=(10, 6))
plt.barh(
    churn_summary_sorted["sales_office_location"],
    churn_summary_sorted["churn_rate"],
    color="skyblue",
)
plt.xlabel("Churn Rate (%)")
plt.ylabel("Sales Office Location")
plt.title("Churn Rate Among Customers Who Abandoned, by Sales Office")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

[Interpretation]

In [0]:
# Summarize churn metrics by sub trade channel
churn_summary = (
    combined_with_customer
    .groupby("customer_sub_trade_channel_description")
    .agg(
        total_customers_who_abandoned=("customer_id", lambda x: x.nunique()),
        churned_customers=("customer_id", lambda x: x[combined_with_customer.loc[x.index, "churned"]].nunique()),
    )
    .reset_index()
)

# Calculate churn rate percentage
churn_summary["churn_rate"] = (
    churn_summary["churned_customers"] / churn_summary["total_customers_who_abandoned"]
) * 100

# Sort summary by churn rate
churn_summary_sorted = churn_summary.sort_values("churn_rate", ascending=False)

# Plot churn rates by sub trade channel
plt.figure(figsize=(10, 6))
plt.barh(
    churn_summary_sorted["customer_sub_trade_channel_description"],
    churn_summary_sorted["churn_rate"],
    color="skyblue",
)
plt.xlabel("Churn Rate (%)")
plt.ylabel("customer_sub_trade_channel_description")
plt.title("Churn Rate Among Customers Who Abandoned, by Sub Trade Channel")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

[Interpretation]

In [0]:
# Sankey diagram of channel switches

# Ensure correct dtypes
ss = switch_summary.copy()
ss["from_channel"] = ss["from_channel"].astype(str)
ss["to_channel"] = ss["to_channel"].astype(str)
ss["count"] = ss["count"].astype(int)

# Build node list and index maps
labels = pd.Index(pd.unique(ss[["from_channel", "to_channel"]].values.ravel("K"))).tolist()
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}

# Map to source/target arrays
sources = ss["from_channel"].map(label_to_idx).tolist()
targets = ss["to_channel"].map(label_to_idx).tolist()
values = ss["count"].tolist()

# Create Sankey
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(label=labels, pad=15, thickness=18),
            link=dict(source=sources, target=targets, value=values),
        )
    ]
)
fig.update_layout(title_text="Post Abandonment Channel Switch Flows", font_size=12, height=500)
fig.show()


[Interpretation]

In [0]:
# Heatmap of from_channel vs to_channel

# Build matrix
ss = switch_summary.copy()
ss["from_channel"] = ss["from_channel"].astype(str)
ss["to_channel"] = ss["to_channel"].astype(str)
ss["count"] = ss["count"].astype(int)

mat = ss.pivot_table(
    index="from_channel",
    columns="to_channel",
    values="count",
    aggfunc="sum",
    fill_value=0,
).astype(int)

# Plot heatmap
plt.figure(figsize=(8, 6))
im = plt.imshow(mat.values, aspect="auto")
plt.colorbar(im, label="Count")

# Axis labels and ticks
plt.xticks(ticks=np.arange(mat.shape[1]), labels=mat.columns, rotation=45, ha="right")
plt.yticks(ticks=np.arange(mat.shape[0]), labels=mat.index)
plt.xlabel("to_channel")
plt.ylabel("from_channel")
plt.title("Post Abandonment Channel Switch Counts Heatmap")

# Optional annotations
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat.iat[i, j]
        if val > 0:
            plt.text(j, i, str(val), ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

[Interpretation]

In [0]:
# Filter switches directed to mycoke360
to_mycoke360 = (
    switch_df[switch_df["to_channel"] == "mycoke360"]["to_channel"]
    .value_counts()
    .reset_index()
)
to_mycoke360.columns = ["channel", "count"]
to_mycoke360["direction"] = "to_mycoke360"

# Filter switches originating from mycoke360
from_mycoke360 = (
    switch_df[switch_df["from_channel"] == "mycoke360"]["from_channel"]
    .value_counts()
    .reset_index()
)
from_mycoke360.columns = ["channel", "count"]
from_mycoke360["direction"] = "from_mycoke360"

# Combine switch data
switch_plot_df = pd.concat([to_mycoke360, from_mycoke360], ignore_index=True)

# Visualize switch counts
plt.figure(figsize=(8, 6))
plt.bar(
    switch_plot_df["direction"],
    switch_plot_df["count"],
    color=["skyblue", "salmon"]
)
plt.ylabel("Number of Switches")
plt.xticks(rotation=45, ha="right")
plt.title("Switches To and From mycoke360")
plt.tight_layout()
plt.show()

[Interpretation]

# Results

Q1 Results - What is the quantitative answer to the question?

Q1 Recommendation(s)
